In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-dp-sh/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

df_bronze=spark.read\
    .option("header",True)\
    .option("inferschema",True)\
    .csv("s3://sportsbar-dp-sh/orders/landing/*.csv")

df_bronze=(df_bronze.withColumn("ingestime",F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size"))

print("The row count:",df_bronze.count())
df_bronze.write .format("delta")\
    .mode("append")\
    .option("header",True)\
    .option("mergeSchema",True)\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


### Moving files from source to processed `directory`

In [0]:
files =dbutils.fs.ls(landing_path)
for files_info in files:
    dbutils.fs.mv(
        files_info.path,
        f"{processed_path}/{files_info.name}"
    )

### Silver Processing Layer

In [0]:
df_silver=spark.table(f"{catalog}.{bronze_schema}.{data_source}")

# 1. Drop rows with null order_id
df_silver=df_silver.filter(F.col("order_id").isNotNull())

# 2. Drop rows with null order_qty
df_silver=df_silver.filter(F.col("order_qty").isNotNull())

# 3. Convert customer_id to string and replace null with 999999
df_silver=df_silver.withColumn("customer_id",
                               F.when(F.col("customer_id").rlike("^[0-9]+$"),F.col("customer_id")).otherwise("999999").cast("string"))

df_silver=df_silver.withColumn("order_placement_date",F.regexp_replace(F.col("order_placement_date"),r"^[A-Za-z]+,\s*",""))

# 4. Convert order_placement_date to date
df_silver = df_silver.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)


# 5. Drop duplicates
df_silver = df_silver.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_silver = df_silver.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_silver.agg(F.min("order_placement_date").alias("min_date"),F.max("order_placement_date").alias("max_date")).display()

In [0]:
df_products=spark.table(f"{catalog}.{silver_schema}.products")
df_joined=df_silver.join(df_products,on="product_id",how="inner")
df_joined=df_joined.select(df_silver["*"],df_products["product_code"])

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", True
    ).option("mergeSchema", True).mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

- ### Gold

In [0]:
df_gold = spark.table(f"{silver_table}")
df_gold=df_gold.select("order_id", F.col("order_placement_date").alias("date"), F.col("customer_id").alias("customer_code"),"product_code", "product_id", "order_qty")

In [0]:
if not(spark.catalog.tableExists(gold_table)):
    print("Creating the New Table")
    df_gold.write.format("delta")\
        .mode("overwrite")\
        .option("mergeSchema",True)\
        .option("delta.enableChangeDataFeed",True)\
        .saveAsTable("gold_table")
else:
    delta_table=DeltaTable.forName(spark,fmcg.gold.orders)
    delta_table.alias("source").merge(df_gold.alias("target"),"target.order_id=source.order_id AND target.date=source.date AND target.customer_code=source=customer_code AND target.product_code=source.product_code").whenMatchedUpdateALL().whenNotMatchedInsertAll().execute()